# 08 | Load into PostgreSQL + PostGIS (SQL)

Loads the eight main tables into a **PostgreSQL** database with the **PostGIS** extension, checks the data quality in SQL, defines the data model (primary and foreign keys) and runs example queries.

| Step | What it does | SQL used |
|---|---|---|
| 1 | Create the schema and tables | `sql/01_create_tables.sql` (DDL) |
| 2 | Load the data from `layers.gpkg` | – |
| 3 | Data quality checks | `GROUP BY` / `HAVING`, `NOT EXISTS`, `FILTER`, `ST_IsValid` |
| 4 | Fix the problems found | `DELETE … USING`, `UPDATE`, `ST_Union`, `ST_MakeValid` |
| 5 | Primary keys, foreign keys and indexes | `sql/02_constraints.sql` |
| 6 | Views for Power BI | `sql/03_powerbi_views.sql` |
| 7 | Example queries | `JOIN`, aggregations, CTE, window functions, spatial functions |

The tables are loaded **without constraints first**, so that problems such as duplicated keys or keys without a match can be found and fixed in SQL before the model is enforced.

**Requirements:** PostgreSQL with the PostGIS extension installed locally, and the Python packages `sqlalchemy`, `psycopg2` and `geoalchemy2`.

*This notebook is an extra step: the CSV export for Power BI (notebook 07) does not depend on it.*

## Setup

In [1]:
import sys
from pathlib import Path

# Make the src/ folder importable when running from the notebooks/ folder
sys.path.append(str(Path.cwd().parent))

import geopandas as gpd
import numpy as np
import pandas as pd

from src.utils import *
import os
from getpass import getpass

from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

### Connection

The password is never written in the code: it is read from the `PGPASSWORD` environment variable or asked when the cell runs. If the database does not exist yet, it is created.

In [2]:
DB_HOST = "localhost"
DB_PORT = 5432
DB_USER = "postgres"
DB_NAME = "geo_reference"
DB_PASSWORD = os.environ.get("PGPASSWORD") or getpass("PostgreSQL password: ")


def make_engine(database):
    url = URL.create("postgresql+psycopg2", username=DB_USER, password=DB_PASSWORD,
                     host=DB_HOST, port=DB_PORT, database=database)
    return create_engine(url)


# Create the database if it does not exist (connecting to the default "postgres" database)
admin_engine = make_engine("postgres").execution_options(isolation_level="AUTOCOMMIT")
with admin_engine.connect() as conn:
    exists = conn.execute(text("SELECT 1 FROM pg_database WHERE datname = :name"), {"name": DB_NAME}).scalar()
    if exists:
        print(f"Database '{DB_NAME}' already exists")
    else:
        conn.execute(text(f'CREATE DATABASE "{DB_NAME}"'))
        print(f"Database '{DB_NAME}' created")

engine = make_engine(DB_NAME)

Database 'geo_reference' already exists


In [3]:
def run_query(query, params=None):
    """Run a SELECT and return the result as a DataFrame."""
    return pd.read_sql(text(query), engine, params=params)


def run_statements(sql):
    """Run one or more SQL statements (DDL, UPDATE, DELETE) in a single transaction."""
    with engine.begin() as conn:
        conn.exec_driver_sql(sql)

## 1. Create the schema and tables

In [4]:
print((SQL_DIR / "01_create_tables.sql").read_text()[:1500], "...")

-- =====================================================================
-- 01 | Create schema and tables (without constraints)
--
-- The tables are created without primary and foreign keys so that the
-- data can be loaded first and checked for quality problems
-- (duplicates, orphan keys, invalid geometries). The constraints are
-- added afterwards in 02_constraints.sql.
-- =====================================================================

CREATE EXTENSION IF NOT EXISTS postgis;
CREATE SCHEMA IF NOT EXISTS reference;

-- Drop in reverse order of dependency, so the script can be run again
DROP TABLE IF EXISTS reference.postal_codes CASCADE;
DROP TABLE IF EXISTS reference.localities CASCADE;
DROP TABLE IF EXISTS reference.municipalities CASCADE;
DROP TABLE IF EXISTS reference.districts CASCADE;
DROP TABLE IF EXISTS reference.nuts3 CASCADE;
DROP TABLE IF EXISTS reference.nuts2 CASCADE;
DROP TABLE IF EXISTS reference.nuts1 CASCADE;
DROP TABLE IF EXISTS reference.nuts0 CASCADE;

-- --

In [5]:
run_statements((SQL_DIR / "01_create_tables.sql").read_text())

run_query("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'reference'
    ORDER BY table_name
""")

,table_name
0,districts
1,localities
2,municipalities
3,nuts0
4,nuts1
5,nuts2
6,nuts3
7,postal_codes


## 2. Load the data

The column names follow the SQL convention (`snake_case`, lower case), so they can be used without quotes in the queries.

In [6]:
common_columns = {"LonCent": "lon_cent", "LatCent": "lat_cent"}

tables_to_load = {
    "nuts0": ("NUTS0_2024_PT_ES", {"NUTS0_ID": "nuts0_id", "NUTS0_Name": "nuts0_name"}),
    "nuts1": ("NUTS1_2024_PT_ES", {"NUTS1_ID": "nuts1_id", "NUTS1_Name": "nuts1_name"}),
    "nuts2": ("NUTS2_2024_PT_ES", {"NUTS2_ID": "nuts2_id", "NUTS2_Name": "nuts2_name"}),
    "nuts3": ("NUTS3_2024_PT_ES", {"NUTS3_ID": "nuts3_id", "NUTS3_Name": "nuts3_name",
                                   "MountainType": "mountain_type", "UrbanType": "urban_type",
                                   "CoastalType": "coastal_type"}),
    "districts": ("Districts2024_PT", {"DistrictID": "district_id", "DistrictName": "district_name",
                                        "AreaHa": "area_ha", "PerimeterKm": "perimeter_km"}),
    "municipalities": ("Municipalities2024_PT", {"MunicipalityID": "municipality_id",
                                                 "MunicipalityName": "municipality_name",
                                                 "AreaHa": "area_ha", "PerimeterKm": "perimeter_km"}),
    "localities": ("Localities2024_PT_ES", {"LocalityID": "locality_id", "MunicipalityID": "municipality_id",
                                            "DistrictID": "district_id", "LocalityName": "locality_name",
                                            "Population2024": "population_2024", "AreaKm2": "area_km2",
                                            "LocalityNameID": "locality_name_id"}),
    "postal_codes": ("PostalCodes2024_PT_ES", {"PostalCode": "postal_code", "NUTS3_ID": "nuts3_id",
                                               "NUTS2_ID": "nuts2_id", "NUTS1_ID": "nuts1_id",
                                               "NUTS0_ID": "nuts0_id", "LocalityID": "locality_id",
                                               "DegreeUrb": "degree_urb"}),
}

# Columns stored as whole numbers in the database (in pandas they are decimals because of the missing values)
integer_columns = ["mountain_type", "urban_type", "coastal_type", "population_2024", "degree_urb"]

source_rows = {}
for table, (layer_name, columns) in tables_to_load.items():
    gdf = load_layer(layer_name).rename(columns={**common_columns, **columns})

    for col in integer_columns:
        if col in gdf.columns:
            gdf[col] = pd.to_numeric(gdf[col], errors="coerce").round().astype("Int64")

    if table == "postal_codes":
        assert (gdf.geom_type == "Point").all(), "Postal codes are expected to be points"
    else:
        gdf = to_multipolygon(gdf)

    gdf.to_postgis(table, engine, schema="reference", if_exists="append", index=False)
    source_rows[table] = len(gdf)
    print(f"Loaded {table}: {len(gdf)} rows")

Loaded nuts0: 2 rows
Loaded nuts1: 10 rows
Loaded nuts2: 28 rows
Loaded nuts3: 85 rows
Loaded districts: 29 rows
Loaded municipalities: 308 rows
Loaded localities: 11224 rows
Loaded postal_codes: 209829 rows


In [7]:
row_counts = run_query("""
    SELECT 'nuts0' AS table_name, COUNT(*) AS rows_in_db FROM reference.nuts0
    UNION ALL SELECT 'nuts1', COUNT(*) FROM reference.nuts1
    UNION ALL SELECT 'nuts2', COUNT(*) FROM reference.nuts2
    UNION ALL SELECT 'nuts3', COUNT(*) FROM reference.nuts3
    UNION ALL SELECT 'districts', COUNT(*) FROM reference.districts
    UNION ALL SELECT 'municipalities', COUNT(*) FROM reference.municipalities
    UNION ALL SELECT 'localities', COUNT(*) FROM reference.localities
    UNION ALL SELECT 'postal_codes', COUNT(*) FROM reference.postal_codes
""")

# Validation: the database has the same number of rows as the source layers
row_counts["rows_in_source"] = row_counts["table_name"].map(source_rows)
row_counts["ok"] = row_counts["rows_in_db"] == row_counts["rows_in_source"]
row_counts

,table_name,rows_in_db,rows_in_source,ok
0,municipalities,308,308,True
1,districts,29,29,True
2,nuts2,28,28,True
3,nuts1,10,10,True
4,nuts0,2,2,True
5,nuts3,85,85,True
6,localities,11224,11224,True
7,postal_codes,209829,209829,True


## 3. Data quality checks

Before defining the keys, three checks are run:
1. **Duplicated or missing keys** – a primary key must be unique and not null.
2. **Orphan foreign keys** – values that do not exist in the table they point to.
3. **Invalid or empty geometries.**

### 3.1 Duplicated and missing keys

In [8]:
primary_keys = {
    "nuts0": "nuts0_id", "nuts1": "nuts1_id", "nuts2": "nuts2_id", "nuts3": "nuts3_id",
    "districts": "district_id", "municipalities": "municipality_id",
    "localities": "locality_id", "postal_codes": "postal_code",
}


def check_keys():
    query = " UNION ALL ".join(
        f"""SELECT '{table}' AS table_name,
                  COUNT(*) AS total_rows,
                  COUNT(*) - COUNT(DISTINCT {key}) - COUNT(*) FILTER (WHERE {key} IS NULL) AS duplicated_rows,
                  COUNT(*) FILTER (WHERE {key} IS NULL) AS null_keys
           FROM reference.{table}"""
        for table, key in primary_keys.items()
    )
    return run_query(query)


check_keys()

,table_name,total_rows,duplicated_rows,null_keys
0,municipalities,308,0,0
1,districts,29,0,0
2,nuts2,28,0,0
3,nuts1,10,0,0
4,nuts0,2,0,0
5,nuts3,85,0,0
6,localities,11224,0,0
7,postal_codes,209829,0,0


Detail of the duplicated keys (if any):

In [9]:
for table, key in primary_keys.items():
    duplicates = run_query(f"""
        SELECT {key}, COUNT(*) AS occurrences
        FROM reference.{table}
        GROUP BY {key}
        HAVING COUNT(*) > 1
        ORDER BY occurrences DESC
        LIMIT 10
    """)
    if len(duplicates) > 0:
        print(f"--- {table}")
        display(duplicates)

### 3.2 Orphan foreign keys

In [10]:
foreign_keys = [
    # (child table, foreign key, parent table, primary key)
    ("postal_codes", "nuts0_id", "nuts0", "nuts0_id"),
    ("postal_codes", "nuts1_id", "nuts1", "nuts1_id"),
    ("postal_codes", "nuts2_id", "nuts2", "nuts2_id"),
    ("postal_codes", "nuts3_id", "nuts3", "nuts3_id"),
    ("postal_codes", "locality_id", "localities", "locality_id"),
    ("localities", "municipality_id", "municipalities", "municipality_id"),
    ("localities", "district_id", "districts", "district_id"),
]


def check_orphans():
    query = " UNION ALL ".join(
        f"""SELECT '{child}.{fk} -> {parent}' AS foreign_key,
                  COUNT(*) FILTER (WHERE c.{fk} IS NOT NULL) AS filled_values,
                  COUNT(*) FILTER (WHERE c.{fk} IS NOT NULL
                                   AND NOT EXISTS (SELECT 1 FROM reference.{parent} p WHERE p.{pk} = c.{fk})) AS orphans
           FROM reference.{child} c"""
        for child, fk, parent, pk in foreign_keys
    )
    return run_query(query)


check_orphans()

,foreign_key,filled_values,orphans
0,postal_codes.nuts0_id -> nuts0,209829,0
1,postal_codes.nuts1_id -> nuts1,209829,0
2,postal_codes.nuts2_id -> nuts2,209829,0
3,postal_codes.nuts3_id -> nuts3,209829,0
4,postal_codes.locality_id -> localities,209829,0
5,localities.municipality_id -> municipalities,11224,8132
6,localities.district_id -> districts,11224,8132


The localities table contains Portugal and Spain, but the CAOP tables only cover Portugal. Orphan keys by country:

In [11]:
run_query("""
    SELECT LEFT(l.locality_id, 2) AS country,
           COUNT(*) AS localities,
           COUNT(*) FILTER (WHERE NOT EXISTS (
               SELECT 1 FROM reference.municipalities m WHERE m.municipality_id = l.municipality_id)) AS no_municipality,
           COUNT(*) FILTER (WHERE NOT EXISTS (
               SELECT 1 FROM reference.districts d WHERE d.district_id = l.district_id)) AS no_district
    FROM reference.localities l
    GROUP BY LEFT(l.locality_id, 2)
    ORDER BY country
""")

,country,localities,no_municipality,no_district
0,ES,8132,8132,8132
1,PT,3092,0,0


### 3.3 Invalid and empty geometries

In [12]:
def check_geometries():
    query = " UNION ALL ".join(
        f"""SELECT '{table}' AS table_name,
                  COUNT(*) FILTER (WHERE NOT ST_IsValid(geometry)) AS invalid,
                  COUNT(*) FILTER (WHERE ST_IsEmpty(geometry)) AS empty
           FROM reference.{table}"""
        for table in primary_keys
    )
    return run_query(query)


check_geometries()

,table_name,invalid,empty
0,nuts0,0,0
1,nuts1,0,0
2,nuts2,0,0
3,nuts3,0,0
4,districts,0,0
5,municipalities,0,0
6,localities,0,0
7,postal_codes,0,0


## 4. Fix the problems found

| Problem | Fix |
|---|---|
| Duplicated district or municipality (the same unit split across two CAOP files, e.g. islands from different groups) | Merge the parts into one row: `ST_Union` of the geometries and sum of area and perimeter |
| Other duplicated keys | Keep the first occurrence (the Power BI model also needs a unique key) |
| Orphan foreign key | Set it to `NULL`: the row is kept, but it is not linked to a non-existent record |
| Invalid geometry | Repair it with `ST_MakeValid` |

Each fix runs without effect when there is nothing to fix.

### 4.1 Merge duplicated districts and municipalities

In [13]:
merge_template = """
CREATE TEMP TABLE merged AS
SELECT {key},
       MIN({name}) AS {name},
       SUM(area_ha) AS area_ha,
       SUM(perimeter_km) AS perimeter_km,
       ST_Multi(ST_Union(geometry)) AS geometry
FROM reference.{table}
WHERE {key} IN (SELECT {key} FROM reference.{table} GROUP BY {key} HAVING COUNT(*) > 1)
GROUP BY {key};

DELETE FROM reference.{table}
WHERE {key} IN (SELECT {key} FROM merged);

-- Centroid calculated in a projected CRS (EPSG:3035) and converted back to EPSG:4326
INSERT INTO reference.{table} ({key}, {name}, area_ha, perimeter_km, lon_cent, lat_cent, geometry)
SELECT {key}, {name}, area_ha, perimeter_km,
       ST_X(ST_Transform(ST_Centroid(ST_Transform(geometry, 3035)), 4326)),
       ST_Y(ST_Transform(ST_Centroid(ST_Transform(geometry, 3035)), 4326)),
       geometry
FROM merged;

DROP TABLE merged;
"""

run_statements(merge_template.format(table="districts", key="district_id", name="district_name"))
run_statements(merge_template.format(table="municipalities", key="municipality_id", name="municipality_name"))

### 4.2 Remove other duplicated keys (keep the first occurrence)

In [14]:
for table in ["nuts0", "nuts1", "nuts2", "nuts3", "localities", "postal_codes"]:
    key = primary_keys[table]
    with engine.begin() as conn:
        result = conn.execute(text(f"""
            DELETE FROM reference.{table} a
            USING reference.{table} b
            WHERE a.{key} = b.{key}
              AND a.ctid > b.ctid
        """))
    print(f"{table}: {result.rowcount} duplicated rows removed")

nuts0: 0 duplicated rows removed
nuts1: 0 duplicated rows removed
nuts2: 0 duplicated rows removed
nuts3: 0 duplicated rows removed
localities: 0 duplicated rows removed
postal_codes: 0 duplicated rows removed


### 4.3 Orphan foreign keys → NULL

In [15]:
for child, fk, parent, pk in foreign_keys:
    with engine.begin() as conn:
        result = conn.execute(text(f"""
            UPDATE reference.{child} c
            SET {fk} = NULL
            WHERE c.{fk} IS NOT NULL
              AND NOT EXISTS (SELECT 1 FROM reference.{parent} p WHERE p.{pk} = c.{fk})
        """))
    print(f"{child}.{fk}: {result.rowcount} values set to NULL")

postal_codes.nuts0_id: 0 values set to NULL
postal_codes.nuts1_id: 0 values set to NULL
postal_codes.nuts2_id: 0 values set to NULL
postal_codes.nuts3_id: 0 values set to NULL
postal_codes.locality_id: 0 values set to NULL
localities.municipality_id: 8132 values set to NULL
localities.district_id: 8132 values set to NULL


### 4.4 Repair invalid geometries

In [16]:
for table in ["nuts0", "nuts1", "nuts2", "nuts3", "districts", "municipalities", "localities"]:
    with engine.begin() as conn:
        result = conn.execute(text(f"""
            UPDATE reference.{table}
            SET geometry = ST_Multi(ST_CollectionExtract(ST_MakeValid(geometry), 3))
            WHERE NOT ST_IsValid(geometry)
        """))
    print(f"{table}: {result.rowcount} geometries repaired")

nuts0: 0 geometries repaired
nuts1: 0 geometries repaired
nuts2: 0 geometries repaired
nuts3: 0 geometries repaired
districts: 0 geometries repaired
municipalities: 0 geometries repaired
localities: 0 geometries repaired


### 4.5 Check again

In [17]:
checks = check_keys().merge(check_geometries(), on="table_name")
checks

,table_name,total_rows,duplicated_rows,null_keys,invalid,empty
0,municipalities,308,0,0,0,0
1,districts,29,0,0,0,0
2,nuts3,85,0,0,0,0
3,nuts2,28,0,0,0,0
4,nuts1,10,0,0,0,0
5,nuts0,2,0,0,0,0
6,localities,11224,0,0,0,0
7,postal_codes,209829,0,0,0,0


In [18]:
check_orphans()

,foreign_key,filled_values,orphans
0,postal_codes.nuts0_id -> nuts0,209829,0
1,postal_codes.nuts1_id -> nuts1,209829,0
2,postal_codes.nuts2_id -> nuts2,209829,0
3,postal_codes.nuts3_id -> nuts3,209829,0
4,postal_codes.locality_id -> localities,209829,0
5,localities.municipality_id -> municipalities,3092,0
6,localities.district_id -> districts,3092,0


## 5. Primary keys, foreign keys and indexes

With the data clean, the model is enforced with constraints (`sql/02_constraints.sql`). From now on, the database refuses duplicated keys and foreign keys without a match. GiST indexes are also created on the geometries, to speed up the spatial queries.

In [19]:
run_statements((SQL_DIR / "02_constraints.sql").read_text())

run_query("""
    SELECT tc.table_name, tc.constraint_type, tc.constraint_name
    FROM information_schema.table_constraints tc
    WHERE tc.table_schema = 'reference'
      AND tc.constraint_type IN ('PRIMARY KEY', 'FOREIGN KEY')
    ORDER BY tc.constraint_type DESC, tc.table_name
""")

,table_name,constraint_type,constraint_name
0,districts,PRIMARY KEY,pk_districts
1,localities,PRIMARY KEY,pk_localities
2,municipalities,PRIMARY KEY,pk_municipalities
3,nuts0,PRIMARY KEY,pk_nuts0
4,nuts1,PRIMARY KEY,pk_nuts1
5,nuts2,PRIMARY KEY,pk_nuts2
6,nuts3,PRIMARY KEY,pk_nuts3
7,postal_codes,PRIMARY KEY,pk_postal_codes
8,localities,FOREIGN KEY,fk_localities_municipalities
9,localities,FOREIGN KEY,fk_localities_districts


## 6. Views for Power BI

Power BI does not need the geometry column, because the maps use the TopoJSON/GeoJSON files. The views in `sql/03_powerbi_views.sql` expose only the attributes. In Power BI: **Get data → PostgreSQL database** (server `localhost`, database `geo_reference`) and select the `v_…` views, as an alternative to the CSV files.

In [20]:
run_statements((SQL_DIR / "03_powerbi_views.sql").read_text())

run_query("""
    SELECT table_name AS view_name
    FROM information_schema.views
    WHERE table_schema = 'reference'
    ORDER BY table_name
""")

,view_name
0,v_districts
1,v_localities
2,v_municipalities
3,v_nuts0
4,v_nuts1
5,v_nuts2
6,v_nuts3
7,v_postal_codes


## 7. Example queries

### 7.1 Postal codes by NUTS 2 region
`JOIN` + aggregation.

In [21]:
run_query("""
    SELECT n2.nuts2_id,
           n2.nuts2_name,
           COUNT(*) AS postal_codes
    FROM reference.postal_codes pc
    JOIN reference.nuts2 n2 ON n2.nuts2_id = pc.nuts2_id
    GROUP BY n2.nuts2_id, n2.nuts2_name
    ORDER BY postal_codes DESC
    LIMIT 10
""")

,nuts2_id,nuts2_name,postal_codes
0,PT11,Norte,68654
1,PT19,Centro (PT),37161
2,PT1A,Grande Lisboa,30107
3,PT1D,Oeste e Vale do Tejo,19796
4,PT1C,Alentejo,13942
5,PT1B,Península de Setúbal,12833
6,PT15,Algarve,7023
7,PT20,Região Autónoma dos Açores,4531
8,PT30,Região Autónoma da Madeira,4354
9,ES41,Castilla y León,2084


### 7.2 Population and density by district (Portugal)
Aggregation of the localities by district, with the official area from the CAOP.

In [22]:
run_query("""
    SELECT d.district_name,
           COUNT(l.locality_id) AS localities,
           SUM(l.population_2024) AS population_2024,
           ROUND((d.area_ha / 100)::numeric, 0) AS area_km2,
           ROUND((SUM(l.population_2024) / (d.area_ha / 100))::numeric, 1) AS inhabitants_per_km2
    FROM reference.districts d
    JOIN reference.localities l ON l.district_id = d.district_id
    GROUP BY d.district_id, d.district_name, d.area_ha
    ORDER BY inhabitants_per_km2 DESC NULLS LAST
""")

,district_name,localities,population_2024,area_km2,inhabitants_per_km2
0,Lisboa,134,2275385,2816.0,808.0
1,Porto,243,1785405,2332.0,765.7
2,Ilha da Madeira,53,245595,758.0,323.8
3,Braga,347,846293,2706.0,312.7
4,Aveiro,147,700787,2801.0,250.2
5,Ilha de São Miguel,64,133288,745.0,179.0
6,Setúbal,55,874806,5214.0,167.8
7,Ilha Terceira,30,53234,400.0,133.0
8,Leiria,110,458605,3506.0,130.8
9,Ilha de Porto Santo,1,5149,43.0,120.6


### 7.3 Largest municipality in each district
CTE + window function (`RANK() OVER (PARTITION BY …)`). The district of each municipality is the first 5 characters of its ID (e.g. `PT_0101` → `PT_01`).

In [23]:
run_query("""
    WITH ranked AS (
        SELECT d.district_name,
               m.municipality_name,
               ROUND((m.area_ha / 100)::numeric, 0) AS area_km2,
               RANK() OVER (PARTITION BY d.district_id ORDER BY m.area_ha DESC) AS area_rank
        FROM reference.municipalities m
        JOIN reference.districts d ON d.district_id = LEFT(m.municipality_id, 5)
    )
    SELECT district_name, municipality_name, area_km2
    FROM ranked
    WHERE area_rank = 1
    ORDER BY area_km2 DESC
""")

,district_name,municipality_name,area_km2
0,Beja,Odemira,1721.0
1,Setúbal,Alcácer do Sal,1500.0
2,Castelo Branco,Castelo Branco,1438.0
3,Évora,Évora,1307.0
4,Bragança,Bragança,1174.0
5,Santarém,Coruche,1116.0
6,Portalegre,Ponte de Sor,840.0
7,Guarda,Sabugal,823.0
8,Vila Real,Montalegre,805.0
9,Faro,Loulé,764.0


### 7.4 Official area vs. area calculated by PostGIS
`ST_Area` on the `geography` type calculates the area on the ellipsoid, in m². The comparison with the official CAOP area validates the geometries.

In [24]:
run_query("""
    SELECT district_name,
           ROUND((area_ha / 100)::numeric, 1) AS official_km2,
           ROUND((ST_Area(geometry::geography) / 1000000)::numeric, 1) AS postgis_km2,
           ROUND((100 * (ST_Area(geometry::geography) / 10000 - area_ha) / area_ha)::numeric, 3) AS difference_pct
    FROM reference.districts
    ORDER BY ABS(ST_Area(geometry::geography) / 10000 - area_ha) / area_ha DESC
""")

,district_name,official_km2,postgis_km2,difference_pct
0,Ilha Terceira,400.3,400.6,0.079
1,Ilha da Graciosa,60.7,60.7,0.061
2,Ilha de São Jorge,243.6,243.8,0.060
3,Ilha do Pico,444.8,445.0,0.046
4,Ilha de Porto Santo,42.7,42.7,0.041
5,Ilha de São Miguel,744.6,744.8,0.035
6,Bragança,6598.5,6596.6,-0.029
7,Ilha do Faial,173.1,173.1,0.026
8,Ilha das Flores,141.0,141.0,0.021
9,Lisboa,2816.1,2815.6,-0.020


### 7.5 Spatial consistency: postal codes outside their NUTS 3 region
Spatial join with `ST_Intersects`. The NUTS boundaries are at a 1:1 million scale (generalised), so a few points near the coast or the borders can fall just outside their polygon.

In [25]:
run_query("""
    SELECT n3.nuts3_id,
           n3.nuts3_name,
           COUNT(*) AS postal_codes,
           COUNT(*) FILTER (WHERE NOT ST_Intersects(pc.geometry, n3.geometry)) AS outside_polygon,
           ROUND(100.0 * COUNT(*) FILTER (WHERE NOT ST_Intersects(pc.geometry, n3.geometry)) / COUNT(*), 2) AS outside_pct
    FROM reference.postal_codes pc
    JOIN reference.nuts3 n3 ON n3.nuts3_id = pc.nuts3_id
    GROUP BY n3.nuts3_id, n3.nuts3_name
    HAVING COUNT(*) FILTER (WHERE NOT ST_Intersects(pc.geometry, n3.geometry)) > 0
    ORDER BY outside_pct DESC
    LIMIT 10
""")

,nuts3_id,nuts3_name,postal_codes,outside_polygon,outside_pct
0,PT1B0,Península de Setúbal,12833,122,0.95
1,PT300,Região Autónoma da Madeira,4354,40,0.92
2,PT200,Região Autónoma dos Açores,4531,33,0.73
3,ES213,Bizkaia,139,1,0.72
4,ES111,A Coruña,402,1,0.25
5,ES511,Barcelona,406,1,0.25
6,ES112,Lugo,478,1,0.21
7,PT191,Região de Aveiro,6952,14,0.20
8,PT193,Região de Leiria,6586,11,0.17
9,PT111,Alto Minho,6004,7,0.12


### 7.6 Neighbouring municipalities
`ST_Touches` finds the municipalities that share a border with a given one. Change the name to try others.

In [26]:
run_query("""
    SELECT b.municipality_name AS neighbour,
           ROUND((ST_Length(ST_Intersection(a.geometry, b.geometry)::geography) / 1000)::numeric, 1) AS shared_border_km
    FROM reference.municipalities a
    JOIN reference.municipalities b
      ON ST_Touches(a.geometry, b.geometry)
    WHERE a.municipality_name = :name
    ORDER BY shared_border_km DESC
""", params={"name": "Aveiro"})

,neighbour,shared_border_km
0,Albergaria-a-Velha,19.7
1,Ílhavo,17.4
2,Murtosa,9.9
3,Oliveira do Bairro,9.1
4,Águeda,7.3
5,Vagos,3.9


### 7.7 Which municipality and NUTS 3 region contain a point?
Point-in-polygon with `ST_Contains`, using the GiST index. Example: Aveiro city centre.

In [27]:
run_query("""
    WITH point AS (
        SELECT ST_SetSRID(ST_MakePoint(:lon, :lat), 4326) AS geom
    )
    SELECT m.municipality_name, n3.nuts3_name, n2.nuts2_name
    FROM point p
    LEFT JOIN reference.municipalities m ON ST_Contains(m.geometry, p.geom)
    LEFT JOIN reference.nuts3 n3 ON ST_Contains(n3.geometry, p.geom)
    LEFT JOIN reference.nuts2 n2 ON n2.nuts2_id = LEFT(n3.nuts3_id, 4)
""", params={"lon": -8.6538, "lat": 40.6405})

,municipality_name,nuts3_name,nuts2_name
0,Aveiro,Região de Aveiro,Centro (PT)


## Summary
* Created the schema and the eight tables in PostgreSQL + PostGIS with typed geometry columns.
* Loaded the data and confirmed that the number of rows matches the source.
* **Data quality in SQL:** checked duplicated and missing keys, orphan foreign keys (by country, since the CAOP only covers Portugal) and invalid geometries.
* **Fixes in SQL:** merged duplicated districts/municipalities with `ST_Union`, removed other duplicates, set orphan keys to `NULL` and repaired invalid geometries.
* Enforced the data model with primary keys, foreign keys and indexes (including spatial GiST indexes), and created views without geometry for Power BI.
* Example queries with joins, aggregations, a CTE with a window function and spatial functions (`ST_Area`, `ST_Intersects`, `ST_Touches`, `ST_Contains`).